<a href="https://colab.research.google.com/github/dhan-t/AcaCHEMi/blob/main/reviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL 1] - IMPORT & SETUP
# ---------------------------------------------------------
# We will create a "Frankenstein" dataset that combines the problems
# found in Airbnb, Netflix, and Pokemon datasets so you can practice
# all techniques at once.
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Create a Dummy Dataset mimicking your Midterm scope
data = {
    'id': [1, 2, 3, 4, 5, 5, 7, 8],  # Duplicate ID '5'
    'content_name': ['Pokemon Red', 'Stranger Things', 'Cozy Apt', 'Iron Man', 'Pikachu', 'Pikachu', 'Luxury Villa', 'Squirtle'],
    'category': ['Game', 'TV Show', 'Airbnb', 'Movie', 'Pokemon', 'Pokemon', 'Airbnb', 'Pokemon'],
    'duration_raw': ['30 hours', '4 Seasons', '2 nights', '120 min', 'Na', 'Na', '5 nights', '45 min'], # Messy Netflix style data
    'release_date': ['1996-02-27', '2016-07-15', '2020/01/01', '2008-05-02', '1996-02-27', '1996-02-27', 'InvalidDate', '1998-09-01'], # Messy dates
    'rating_score': [90, 85, 4.5, 80, 50, 50, 4.8, 1000], # 1000 is an Outlier (Pokemon style)
    'views': [10000, 500000, 200, 300000, 5000, 5000, 150, 4000]
}

df = pd.DataFrame(data)
print("--- RAW DATA ---")
print(df)

In [ ]:
# [CELL 2] - STRUCTURING (From Exercise 1: Airbnb)
# ---------------------------------------------------------
# Goal: Fix types and memory usage.
# ---------------------------------------------------------

# 1. Convert Object to Category (Optimizes memory)
# You did this in Exercise 1 for 'neighbourhood_group'
cols_to_cat = ['category']
for col in cols_to_cat:
    df[col] = df[col].astype('category')

# 2. Convert Object to Datetime
# You did this in Exercise 1 for 'last_review'.
# errors='coerce' turns "InvalidDate" into NaT (Not a Time)
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

print("\n--- TYPES AFTER STRUCTURING ---")
print(df.dtypes)

In [ ]:
# [CELL 3] - CLEANING (From Exercise 1 & 3)
# ---------------------------------------------------------
# Goal: Handle Duplicates and Missing Values.
# ---------------------------------------------------------

# 1. Handle Duplicates (Airbnb style)
print(f"\nDuplicates found: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)

# 2. Handle Missing Values (Netflix style)
print("\nMissing values per column:")
print(df.isnull().sum())

# Strategy: Fill missing 'duration_raw' with a placeholder
df['duration_raw'] = df['duration_raw'].fillna('0 min')

# Strategy: Drop rows where Date is broken (NaT)
df_clean = df.dropna(subset=['release_date']).copy()

In [ ]:
# [CELL 4] - FEATURE ENGINEERING (From Exercise 2: Netflix)
# ---------------------------------------------------------
# Goal: Extract numbers from messy text (e.g., "90 min" -> 90)
# This mimics the custom function you wrote in Exercise 2.
# ---------------------------------------------------------

def parse_duration(val):
    val = str(val).lower()
    if 'min' in val:
        return int(val.split(' ')[0])
    elif 'season' in val:
        # Transforming seasons to approx minutes (like you did in Ex 2)
        return int(val.split(' ')[0]) * 400
    elif 'night' in val:
        # Transforming nights (Airbnb style) to arbitrary minutes
        return int(val.split(' ')[0]) * 1440
    else:
        return 0

df_clean['duration_mins'] = df_clean['duration_raw'].apply(parse_duration)

print("\n--- DATA AFTER ENGINEERING ---")
print(df_clean[['content_name', 'duration_raw', 'duration_mins']].head())

In [ ]:
# [CELL 5] - EDA & STATS (From Exercise 3: Pokemon)
# ---------------------------------------------------------
# Goal: Calculate Mean, Median, Std, Quantiles.
# ---------------------------------------------------------

# 1. Summary Stats
print("\n--- STATS (Describe) ---")
print(df_clean['rating_score'].describe())

# 2. Specific Stat checks (Exercise 3 asked for these)
mean_val = df_clean['rating_score'].mean()
median_val = df_clean['rating_score'].median()
std_val = df_clean['rating_score'].std()

print(f"\nMean Rating: {mean_val}")
print(f"Median Rating: {median_val}")
print(f"Std Dev Rating: {std_val}")

# 3. Quantiles (25th and 75th percentile)
q25 = df_clean['rating_score'].quantile(0.25)
q75 = df_clean['rating_score'].quantile(0.75)
print(f"25th Percentile: {q25}")
print(f"75th Percentile: {q75}")

# 4. Outlier Detection (Pokemon style)
# Notice the max is 1000? That's an outlier.
outliers = df_clean[df_clean['rating_score'] > 100]
print("\n--- OUTLIERS DETECTED ---")
print(outliers[['content_name', 'rating_score']])

# Filter out the outlier for the next steps
df_final = df_clean[df_clean['rating_score'] <= 100].copy()

In [ ]:
# [CELL 6] - SCALING (From Exercise 2: Netflix)
# ---------------------------------------------------------
# Goal: Compare MinMax vs Standard Scaler.
# ---------------------------------------------------------

# Let's scale 'duration_mins' and 'rating_score'
features = df_final[['duration_mins', 'rating_score']]

# 1. Min-Max (0 to 1)
scaler_mm = MinMaxScaler()
df_final[['dur_minmax', 'rating_minmax']] = scaler_mm.fit_transform(features)

# 2. Standard Scaler (Z-Score)
scaler_std = StandardScaler()
df_final[['dur_zscore', 'rating_zscore']] = scaler_std.fit_transform(features)

print("\n--- SCALING COMPARISON ---")
print(df_final[['duration_mins', 'dur_minmax', 'dur_zscore']].head())

In [ ]:
# [CELL 7] - DISCRETIZATION (From Exercise 2: Netflix)
# ---------------------------------------------------------
# Goal: Bin continuous data into categories (Low/Med/High).
# ---------------------------------------------------------

# Binning 'rating_score' into 3 bins: Low, Mid, High
# We use pd.cut for Equal Width binning
df_final['rating_category'] = pd.cut(
    df_final['rating_score'],
    bins=[0, 50, 80, 100],
    labels=['Low', 'Mid', 'High']
)

print("\n--- DISCRETIZATION ---")
print(df_final[['content_name', 'rating_score', 'rating_category']])

In [ ]:
# [CELL 8] - VISUALIZATION (From Exercise 3)
# ---------------------------------------------------------
# Goal: Visualizing relationships and distributions.
# ---------------------------------------------------------

plt.figure(figsize=(10, 4))

# 1. Boxplot (Good for outliers like in Pokemon Ex 3)
plt.subplot(1, 2, 1)
sns.boxplot(y=df_clean['rating_score']) # Using df_clean to show the outlier
plt.title('Boxplot Showing Outlier (1000)')

# 2. Histogram (Distribution of Scaled Data)
plt.subplot(1, 2, 2)
sns.histplot(df_final['dur_zscore'], kde=True)
plt.title('Distribution of Standardized Duration')

plt.tight_layout()
plt.show()

In [ ]:
# [CELL 9] - TOPIC: INDEXING & SELECTION (The Navigation Tools)
# ---------------------------------------------------------
# Goal: Practice .loc, .iloc, set_index, and idxmax.
# Reference: Exercise 3 (Pokemon) & General Pandas Skills
# ---------------------------------------------------------

print("\n--- INDEXING PRACTICE ---")

# 1. Setting an Index
# Right now, our rows are numbered 0, 1, 2...
# Let's make 'content_name' the official label (The Index).
df_indexed = df_final.set_index('content_name')

print("Dataframe with 'content_name' as Index:")
print(df_indexed[['rating_score', 'views']].head())

# 2. Using .loc (Label Location)
# "Find the data for 'Pikachu'"
# Note: Since we have duplicates of 'Pikachu' in our dummy data,
# this might return multiple rows!
try:
    pikachu_data = df_indexed.loc['Pikachu']
    print(f"\nData for Pikachu (.loc):\n{pikachu_data}")
except KeyError:
    print("\nPikachu not found in this filtered dataset.")

# 3. Using .iloc (Integer Location)
# "Find the data for the 3rd row (index 2)"
third_row = df_indexed.iloc[2]
print(f"\nData for 3rd Row (.iloc):\n{third_row}")

# 4. Advanced Selection: idxmax()
# Question: Which content has the highest views?
# idxmax returns the INDEX (Name) of the max value, not the value itself.
most_viewed_content = df_indexed['views'].idxmax()
max_views = df_indexed['views'].max()

print(f"\nThe most viewed content is '{most_viewed_content}' with {max_views} views.")

# 5. Conditional Assignment (Labeling based on logic)
# Let's make a new label 'Success_Status'
# If views > 10000 -> 'Hit', else 'Flop'
df_indexed.loc[df_indexed['views'] > 10000, 'Success_Status'] = 'Hit'
df_indexed.loc[df_indexed['views'] <= 10000, 'Success_Status'] = 'Flop'

print("\nNew Labels Created:")
print(df_indexed[['views', 'Success_Status']].head())

In [ ]:
# [CELL 10] - TOPIC: ADVANCED LOGIC & MERGING (The Likely Missing Pieces)
# ---------------------------------------------------------
# Goal: Practice Lambda functions and joining dataframes.
# ---------------------------------------------------------

# 1. Lambda Functions
# Think of this as a "Disposable Function" you write in one line.
# Example: Create a 'flag' if views are high.
print("--- LAMBDA FUNCTION PRACTICE ---")

# "For every x (row value), if x > 1000 return 'Viral', else 'Normal'"
df_indexed['Viral_Status'] = df_indexed['views'].apply(lambda x: 'Viral' if x > 1000 else 'Normal')
print(df_indexed[['views', 'Viral_Status']].head())


# 2. Merging Data (Joining Tables)
# Imagine you have a separate list of 'Content Creators' for these shows.
print("\n--- MERGING DATASETS ---")

creators_data = {
    'content_name': ['Pokemon Red', 'Stranger Things'],
    'Creator': ['Game Freak', 'Duffer Bros']
}
df_creators = pd.DataFrame(creators_data)

# We want to add the 'Creator' column to our main dataframe.
# We match them up using 'content_name'.
# how='left' means "Keep all our main shows, just add info if found."
df_merged = pd.merge(df_indexed.reset_index(), df_creators, on='content_name', how='left')

print(df_merged[['content_name', 'Creator']].head())

In [ ]:
# [CELL 11] - TOPIC: ADVANCED INDEXING & FILTERING
# ---------------------------------------------------------
# Goal: Master .loc, Boolean Masks, and Labeling
# Reference: Exercise 3 (Pokemon) specifically used these!
# ---------------------------------------------------------

# Let's use our 'df_final' from the previous code (Pokemon/Netflix mix)

print("--- 1. BOOLEAN INDEXING (THE MASK) ---")
# We want only 'High' rated content that is ALSO a 'Game'
# This uses the logic: (Condition A) & (Condition B)
mask = (df_final['rating_category'] == 'High') & (df_final['category'] == 'Pokemon')

print("How many items fit our criteria?", mask.sum())
print("\nThe items:")
# We apply the mask to the dataframe
high_rated_pokemon = df_final[mask]
print(high_rated_pokemon[['content_name', 'rating_score', 'category']])


print("\n--- 2. FANCY INDEXING (nlargest) ---")
# Your Exercise 3 used .nlargest() to find the fastest Pokemon.
# Let's find the top 3 items with the highest duration.
top_3_longest = df_final.nlargest(3, 'duration_mins')
print("Top 3 Longest Content:")
print(top_3_longest[['content_name', 'duration_mins']])


print("\n--- 3. LABELING & RENAMING ---")
# Sometimes 'making labels' means renaming columns to be clearer.
# Let's rename 'content_name' to 'Title' and 'rating_score' to 'Score'
df_labeled = df_final.rename(columns={
    'content_name': 'Title',
    'rating_score': 'Score'
})

# We can also 'make labels' by setting a specific column as the Index
df_labeled.set_index('Title', inplace=True)

print(df_labeled[['Score', 'category']].head())


print("\n--- 4. ACCESSING BY INDEX (.loc vs .iloc) ---")
# NOW that we set 'Title' as the index, we can search by Name directly.

# .loc is for LABELS (Names)
print("Using .loc for 'Iron Man':")
print(df_labeled.loc['Iron Man'])

# .iloc is for INTEGERS (Position)
# "Give me the very first row, whatever it is named"
print("\nUsing .iloc[0] (First Row):")
print(df_labeled.iloc[0])

In [ ]:
# [CELL 12] - TOPIC: CENTRAL TENDENCY, DISPERSION, SHAPE
# ---------------------------------------------------------
# Goal: Calculate stats and visualize the "Shape" of data.
# ---------------------------------------------------------

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Let's reuse our Pokemon-style data for this
# I'll add more data points to make the stats interesting
stats_data = {
    'Name': ['Pikachu', 'Bulbasaur', 'Charmander', 'Mewtwo', 'Snorlax', 'Magikarp', 'Gengar', 'Dragonite', 'Eevee', 'Jigglypuff'],
    'HP': [35, 45, 39, 106, 160, 20, 60, 91, 55, 115],
    'Attack': [55, 49, 52, 110, 110, 10, 65, 134, 55, 45],
    'Defense': [40, 49, 43, 90, 65, 55, 60, 95, 50, 20],
    'Speed': [90, 45, 65, 130, 30, 80, 110, 80, 55, 20]
}

df_stats = pd.DataFrame(stats_data)
df_stats.set_index('Name', inplace=True) # Set Name as index (Labeling)

print("--- 1. CENTRAL TENDENCY (The Center) ---")
# Mean vs Median (Check for outliers!)
mean_hp = df_stats['HP'].mean()
median_hp = df_stats['HP'].median()
mode_atk = df_stats['Attack'].mode()[0]

print(f"Mean HP: {mean_hp:.2f}")
print(f"Median HP: {median_hp:.2f}")
print(f"Mode Attack: {mode_atk}")
print(f"Difference (Mean - Median): {mean_hp - median_hp:.2f}")
# If Mean > Median, it's usually Right Skewed (Positive Skew)


print("\n--- 2. DISPERSION (The Spread) ---")
# Range
hp_range = df_stats['HP'].max() - df_stats['HP'].min()

# Standard Deviation
std_hp = df_stats['HP'].std()

# Quantiles (Percentiles)
q1 = df_stats['HP'].quantile(0.25)
q3 = df_stats['HP'].quantile(0.75)
iqr = q3 - q1

print(f"Range of HP: {hp_range}")
print(f"Std Dev of HP: {std_hp:.2f}")
print(f"IQR of HP: {iqr} (Middle 50% range)")


print("\n--- 3. SHAPE (The Lean) ---")
# Skewness
skew_val = df_stats['HP'].skew()
kurt_val = df_stats['HP'].kurt()

print(f"Skewness: {skew_val:.2f}")
if skew_val > 0.5:
    print("-> The data is Positively Skewed (Tail on the Right)")
elif skew_val < -0.5:
    print("-> The data is Negatively Skewed (Tail on the Left)")
else:
    print("-> The data is fairly Symmetrical")


print("\n--- 4. INDEXING WITH STATS (The 'Who') ---")
# This is the "idx" part you asked about.
# Question: Who has the MAX HP?

# .max() gives the VALUE (160)
max_hp_val = df_stats['HP'].max()
# .idxmax() gives the NAME (Snorlax) -> This is the Index Label
strongest_pokemon = df_stats['HP'].idxmax()

print(f"Highest HP Value: {max_hp_val}")
print(f"Pokemon with Highest HP: {strongest_pokemon}")

# Finding the 'Who' for the minimum
weakest_pokemon = df_stats['HP'].idxmin()
print(f"Pokemon with Lowest HP: {weakest_pokemon}")

# Finding specific stats for a specific label
# "What is Mewtwo's Speed?"
mewtwo_speed = df_stats.loc['Mewtwo', 'Speed']
print(f"Mewtwo's Speed: {mewtwo_speed}")


print("\n--- 5. VISUALIZING SHAPE ---")
# A histogram is the best way to see Shape
plt.figure(figsize=(6, 4))
sns.histplot(df_stats['HP'], kde=True, bins=5)
plt.axvline(mean_hp, color='red', linestyle='--', label='Mean')
plt.axvline(median_hp, color='green', linestyle='-', label='Median')
plt.title(f"Shape of HP Distribution (Skew: {skew_val:.2f})")
plt.legend()
plt.show()

In [ ]:
# [CELL 13] - TOPIC: GROUPBY & AGGREGATION
# ---------------------------------------------------------
# Goal: Compare groups using Split-Apply-Combine.
# Reference: Exercise 1 (Airbnb Neighbourhoods) & Ex 3 (Pokemon Types)
# ---------------------------------------------------------

# Let's use our 'df_final' (The Pokemon/Content dataset)

print("--- 1. SIMPLE GROUPBY (The 'One Question' Approach) ---")
# Question: Which Category has the highest Average Rating?
# Logic: Split by 'category' -> Calculate Mean of 'rating_score'
avg_rating = df_final.groupby('category')['rating_score'].mean()

print("Average Rating by Category:")
print(avg_rating)
# Note: The result looks like a list where 'category' is the Index.


print("\n--- 2. ADVANCED AGGREGATION (The 'Multiple Questions' Approach) ---")
# Question: Give me the Count, Mean, and Max of 'views' for each category.
# We use .agg() to pass a list of math functions.
group_stats = df_final.groupby('category')['views'].agg(['count', 'mean', 'max'])

print("Detailed View Stats:")
print(group_stats)


print("\n--- 3. MULTI-LEVEL GROUPING (Drilling Deeper) ---")
# Question: Group by 'Category' AND 'Rating Level' (Low/Mid/High)
# This creates a specific bucket like: "Pokemon that are High Rated"
multi_group = df_final.groupby(['category', 'rating_category'])['views'].mean()

print("Average Views by Category AND Rating:")
print(multi_group)


print("\n--- 4. RESET INDEX (Making it look like a normal Excel sheet) ---")
# When you group, the 'names' (Pokemon, Movie) become the Index (labels).
# Usually, you want them back as normal columns so you can work with them.
flat_table = group_stats.reset_index()

print("Clean Table (After reset_index):")
print(flat_table)